# Cleaning source_crm Data

## 1.1 Customer table

In [0]:
%python
from pyspark.sql.functions import trim, to_date, col,current_timestamp



In [0]:
customers_df = spark.table("bronze.customers") 

In [0]:
customers_silver_df = customers_df \
   .withColumn("cst_key", trim("cst_key")) \
    .withColumn("cst_firstname", trim("cst_firstname")) \
    .withColumn("cst_lastname", trim("cst_lastname")) \
    .withColumn("cst_marital_status", trim("cst_marital_status")) \
    .withColumn("cst_gndr", trim("cst_gndr")) \
    .withColumn("created_ts", current_timestamp()) \
    .filter("cst_id IS NOT NULL AND cst_key IS NOT NULL") \
 

In [0]:
customers_silver_df = customers_df \
    .withColumnRenamed("cst_id", "id") \
    .withColumnRenamed("cst_key", "key") \
    .withColumnRenamed("cst_firstname", "first_name") \
    .withColumnRenamed("cst_lastname", "last_name") \
    .withColumnRenamed("cst_marital_status", "marital_status") \
    .withColumnRenamed("cst_gndr", "gender") \
    .withColumnRenamed("cst_create_date", "date")

In [0]:
customers_silver_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.customers")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("id").orderBy(col("date").desc())

customers_dedup_df = customers_silver_df \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter("row_num = 1") \
    .drop("row_num")



In [0]:
print("Before deduplication:", customers_silver_df.count())
print("After deduplication:", customers_dedup_df.count())


In [0]:
%sql
SELECT * FROM silver.customers ;

## 1.2 Products table

In [0]:
products_df = spark.table("bronze.products") 

In [0]:
products_silver_df = products_df \
    .withColumn("prd_key", trim("prd_key")) \
    .withColumn("prd_nm", trim("prd_nm")) \
    .withColumn("prd_line", trim("prd_line")) \
    .withColumn("created_ts", current_timestamp()) \
    .filter("prd_id IS NOT NULL AND prd_key IS NOT NULL AND prd_cost IS NOT NULL")


In [0]:
products_silver_df = products_df \
    .withColumnRenamed("prd_id", "id") \
    .withColumnRenamed("prd_key", "product_key") \
    .withColumnRenamed("prd_nm", "name") \
    .withColumnRenamed("prd_cost", "cost") \
    .withColumnRenamed("prd_line", "line") \
    .withColumnRenamed("prd_start_dt", "start_date") \
    .withColumnRenamed("prd_end_dt", "end_date")

In [0]:
products_silver_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.products")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("id").orderBy(col("start_date").desc())

products_dedup_df = products_silver_df \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter("row_num = 1") \
    .drop("row_num")


In [0]:
print("Before deduplication:", products_silver_df.count())
print("After deduplication:", products_dedup_df.count())


In [0]:
%sql
SELECT * FROM silver.products LIMIT 10;

##  1.3 Sales table

In [0]:
sales_df = spark.table("bronze.sales") 

In [0]:
sales_silver_df = sales_df \
    .withColumn("sls_ord_num", trim("sls_ord_num")) \
    .withColumn("sls_prd_key", trim("sls_prd_key")) \
    .withColumn("sls_order_dt", to_date(col("sls_order_dt").cast("string"), "yyyyMMdd")) \
    .withColumn("sls_ship_dt", to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd")) \
    .withColumn("sls_due_dt", to_date(col("sls_due_dt").cast("string"), "yyyyMMdd")) \
    .withColumn("created_ts", current_timestamp()) \
    .filter("sls_ord_num IS NOT NULL AND sls_prd_key IS NOT NULL")

In [0]:
    sales_silver_df = sales_df \
    .withColumnRenamed("sls_ord_num", "order_number") \
    .withColumnRenamed("sls_prd_key", "product_key") \
    .withColumnRenamed("sls_cust_id", "customer_id") \
    .withColumnRenamed("sls_order_dt", "order_date") \
    .withColumnRenamed("sls_ship_dt", "ship_date") \
    .withColumnRenamed("sls_due_dt", "due_date") \
    .withColumnRenamed("sls_sales", "sales_amount") \
    .withColumnRenamed("sls_quantity", "quantity") \
    .withColumnRenamed("sls_price", "price")

In [0]:
sales_silver_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.sales")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col


window_spec = Window.partitionBy("order_number").orderBy(col("ship_date").desc())


In [0]:
sales_dedup_df = sales_silver_df \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter("row_num = 1") \
    .drop("row_num")


In [0]:
print("Before deduplication:", sales_silver_df.count())
print("After deduplication:", sales_dedup_df.count())

In [0]:
%sql
SELECT * FROM silver.sales LIMIT 10;

# Cleaning source_erp Data

## 2.1 Customers_erp table

In [0]:
customers_erp_df = spark.table("bronze.customers_erp") 

In [0]:
customers_erp_silver_df = customers_erp_df \
    .withColumn("cid", trim("cid")) \
    .withColumn("gen", trim("gen")) \
    .withColumn("created_ts", current_timestamp()) \
    .filter("cid IS NOT NULL AND gen IS NOT NULL")

In [0]:
customers_erp_silver_df = customers_erp_df \
    .withColumnRenamed("CID", "customer_id") \
    .withColumnRenamed("BDATE", "birth_date") \
    .withColumnRenamed("GEN", "gender")


In [0]:
customers_erp_silver_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.customers_erp")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("customer_id").orderBy(col("birth_date").desc())

customers_erp_dedup_df = customers_erp_silver_df \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter("row_num = 1") \
    .drop("row_num")

In [0]:
print("Before deduplication:", customers_erp_silver_df.count())
print("After deduplication:", customers_erp_dedup_df.count())

In [0]:
%sql
SELECT * FROM silver.customers_erp Limit 10;

## 2.2Locations table

In [0]:
locations_df = spark.table("bronze.locations") 

In [0]:
locations_silver_df = locations_df \
    .withColumn("cid", trim("cid")) \
    .withColumn("cntry", trim("cntry")) \
    .withColumn("created_ts", current_timestamp()) \
    .filter("cid IS NOT NULL AND cntry IS NOT NULL")

In [0]:
locations_silver_df = locations_df \
    .withColumnRenamed("cid", "customer_id") \
    .withColumnRenamed("cntry", "country")


In [0]:
locations_silver_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.locations")


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("customer_id").orderBy(col("country").desc())

locations_dedup_df = locations_silver_df \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter("row_num = 1") \
    .drop("row_num")




In [0]:
print("Before deduplication:", locations_silver_df.count())
print("After deduplication:", locations_dedup_df.count())

In [0]:
%sql
SELECT * FROM silver.locations LIMIT 10;

## 2.3 Categories table

In [0]:
categories_df = spark.table("bronze.categories") 

In [0]:
categories_silver_df = categories_df \
    .withColumn("id", trim("id")) \
    .withColumn("cat", trim("cat")) \
    .withColumn("subcat", trim("subcat")) \
    .withColumn("maintenance", trim("maintenance")) \
    .withColumn("created_ts", current_timestamp()) \
    .filter("id IS NOT NULL AND cat IS NOT NULL")

In [0]:
categories_silver_df = categories_df \
    .withColumnRenamed("cat", "category") \
    .withColumnRenamed("subcat", "subcategory") \
    .withColumnRenamed("id", "id") \
    .withColumnRenamed("maintenance","maintenance")
    

In [0]:
categories_silver_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.categories")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("id").orderBy(col("maintenance").desc())

categories_dedup_df = categories_silver_df \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter("row_num = 1") \
    .drop("row_num")


In [0]:
print("Before deduplication:", categories_silver_df.count())
print("After deduplication:", categories_dedup_df.count())

In [0]:
%sql
SELECT * FROM silver.categories ;